In [2]:
import numpy as np
import cv2
import sys
import time
from numba import jit

# ==========================================
# 1. Documentation & Input/Output Format
# ==========================================
"""
Algorithm: Seam Carving for Content-Aware Image Resizing

Input Format:
    - A standard image file (JPG, PNG).
    - Represented internally as a NumPy array of shape (Height, Width, Channels).
    - Values are integers 0-255 (uint8).

Output Format:
    - A processed image file.
    - Represented as a NumPy array of shape (Height, Target_Width, Channels).
    - The output preserves the height but reduces the width by removing low-energy seams.
"""

# ==========================================
# 2. Energy Calculation
# ==========================================
def compute_dual_gradient_energy(img):
    """
    E(x,y) = sqrt(dx^2 + dy^2).
    """
    # Convert to float64 to ensure precision during squaring
    img = img.astype(np.float64)
    
    # [OPTIMIZATION] Compute gradients along Y (axis 0) and X (axis 1) simultaneously.
    # np.gradient automatically handles boundaries using forward/backward difference.
    # It returns a tuple: (gradient_along_axis_0, gradient_along_axis_1)
    gradients = np.gradient(img, axis=(0, 1))
    grad_y, grad_x = gradients[0], gradients[1]
    
    # Square the gradients
    energy_sq = np.square(grad_x) + np.square(grad_y)
    
    # If RGB, sum energy across color channels to get a single 2D energy map
    if img.ndim == 3:
        energy_map = np.sum(energy_sq, axis=2)
    else:
        energy_map = energy_sq
        
    return np.sqrt(energy_map)

# ==========================================
# 3. Optimal Seam Finding (The DP Core)
# ==========================================
@jit(nopython=True, cache=True)
def find_optimal_seam(energy_map):
    """
    Identifies the vertical seam with the lowest total energy.
    
    Implements the Dynamic Programming approach manually.
    """
    H, W = energy_map.shape
    
    # [MEMOIZATION]
    # M[i, j] acts as our DP table/memoization structure.
    # It stores the minimum cost to reach pixel (i, j) from the top row.
    M = energy_map.copy()
    
    # 'backtrack' table stores the decision made at each step to reconstruct the path later.
    backtrack = np.zeros((H, W), dtype=np.int8)
    
    # [BASE CASE]
    # The top row (row 0) has no parents. 
    # The cost to reach any pixel in row 0 is simply its own energy.
    # M[0, :] is already initialized to energy_map[0, :] via the copy() above.
    
    # [RECURSIVE RELATION]
    # Iterate from the second row to the bottom (Bottom-Up DP)
    for i in range(1, H):
        for j in range(W):
            # We look at the 3 possible parents in the row above (i-1):
            # Top-Left (j-1), Top-Middle (j), Top-Right (j+1)
            
            # Default parent: Top-Middle
            min_prev_energy = M[i-1, j]
            offset = 0
            
            # Check Top-Left
            if j > 0:
                if M[i-1, j-1] < min_prev_energy:
                    min_prev_energy = M[i-1, j-1]
                    offset = -1
            
            # Check Top-Right
            if j < W - 1:
                if M[i-1, j+1] < min_prev_energy:
                    min_prev_energy = M[i-1, j+1]
                    offset = 1
            
            # Apply the recurrence: Cost = Current_Energy + Min_Parent_Cost
            M[i, j] += min_prev_energy
            backtrack[i, j] = offset
            
    # --- Backtracking (Path Reconstruction) ---
    seam = np.zeros(H, dtype=np.int64)
    
    # 1. Find the end of the minimal seam in the last row
    min_val = M[H-1, 0]
    min_idx = 0
    for j in range(1, W):
        if M[H-1, j] < min_val:
            min_val = M[H-1, j]
            min_idx = j
            
    seam[H-1] = min_idx
    
    # 2. Trace path upwards using the backtrack table
    curr_col = min_idx
    for i in range(H-2, -1, -1):
        offset = backtrack[i+1, curr_col]
        curr_col += offset
        seam[i] = curr_col
        
    return seam

# ==========================================
# 4. Seam Removal
# ==========================================
@jit(nopython=True, cache=True)
def remove_seam(img, seam):
    """
    Removes the seam from the image by shifting pixels left.
    """
    H, W, C = img.shape
    new_img = np.zeros((H, W-1, C), dtype=img.dtype)
    
    for i in range(H):
        remove_idx = seam[i]
        # Shift pixels left to overwrite the seam
        new_img[i, :remove_idx] = img[i, :remove_idx]
        new_img[i, remove_idx:] = img[i, remove_idx+1:]
        
    return new_img

# ==========================================
# 5. Main Driver
# ==========================================
def seam_carving_driver(img, target_width):
    current_img = img
    initial_width = current_img.shape[1]
    
    print(f"Starting Seam Carving...")
    print(f"Original Size: {current_img.shape[:2]}")
    print(f"Target Width:  {target_width}")
    
    start_time = time.time()
    
    while current_img.shape[1] > target_width:
        # Step 1: Compute Energy Map
        energy_map = compute_dual_gradient_energy(current_img)
        
        # Step 2: Solve DP Problem
        seam = find_optimal_seam(energy_map)
        
        # Step 3: Modify Image
        current_img = remove_seam(current_img, seam)
        
        # Logging
        removed_count = initial_width - current_img.shape[1]
        if removed_count % 10 == 0:
            print(f"Removed {removed_count} seams...")
            
    total_time = time.time() - start_time
    print(f"Finished in {total_time:.2f} seconds.")
    return current_img

# ==========================================
# 6. Execution & Test Block
# ==========================================
if __name__ == "__main__":
    # --- CONFIGURATION ---
    INPUT_FILENAME = "Conan.jpg"
    OUTPUT_FILENAME = "output_image.jpg"
    PIXELS_TO_REMOVE = 500  # Shrink width by 500 pixels by default, but you are free to modify.
    
    # Try to load a real image
    real_img = cv2.imread(INPUT_FILENAME)
    
    if real_img is not None:
        print(f"--- Processing Real Image: {INPUT_FILENAME} ---")
        target_w = real_img.shape[1] - PIXELS_TO_REMOVE
        
        if target_w <= 0:
            print("Error: Target width must be > 0.")
        else:
            result_img = seam_carving_driver(real_img, target_w)
            cv2.imwrite(OUTPUT_FILENAME, result_img)
            print(f"Result saved to {OUTPUT_FILENAME}")
        
    else:
        print(f"--- Image '{INPUT_FILENAME}' not found. Running Synthetic Validity Check ---")
        
        # Create Synthetic Test Case (5x5)
        # Setup: Indices 0,1,3 = 100 (Medium), Index 2 = 255 (High), Index 4 = 0 (Low absolute, but high contrast)
        test_vals = np.array([
            [100, 100, 255, 100, 0],
            [100, 100, 255, 100, 0],
            [100, 100, 255, 100, 0],
            [100, 100, 255, 100, 0],
            [100, 100, 255, 100, 0]
        ], dtype=np.uint8)
        
        # Stack to make RGB
        test_img_rgb = np.stack((test_vals,)*3, axis=-1)
        
        print("Test Image (Energy should be lowest at perfectly flat column 0):")
        print(test_img_rgb[:,:,0])
        
        # Run 1 iteration of the algorithm
        energy = compute_dual_gradient_energy(test_img_rgb)
        seam = find_optimal_seam(energy)
        
        print("\nCalculated Energy Map (Row 0):", np.round(energy[0], 1))
        print("Identified Seam Path (Indices):", seam)
        
        # Verification Logic (CORRECTED)
        if 2 in seam:
            print("[FAIL] Algorithm crossed the high energy center (Index 2).")
        elif np.all(seam == 0):
            print("[PASS] Algorithm successfully followed the zero-gradient path (Index 0).")
        else:
            print(f"[FAIL] Algorithm chose an unexpected path: {seam}")

--- Processing Real Image: Conan.jpg ---
Starting Seam Carving...
Original Size: (2088, 3341)
Target Width:  2841
Removed 10 seams...
Removed 20 seams...
Removed 30 seams...
Removed 40 seams...
Removed 50 seams...
Removed 60 seams...
Removed 70 seams...
Removed 80 seams...
Removed 90 seams...
Removed 100 seams...
Removed 110 seams...
Removed 120 seams...
Removed 130 seams...
Removed 140 seams...
Removed 150 seams...
Removed 160 seams...
Removed 170 seams...
Removed 180 seams...
Removed 190 seams...
Removed 200 seams...
Removed 210 seams...
Removed 220 seams...
Removed 230 seams...
Removed 240 seams...
Removed 250 seams...
Removed 260 seams...
Removed 270 seams...
Removed 280 seams...
Removed 290 seams...
Removed 300 seams...
Removed 310 seams...
Removed 320 seams...
Removed 330 seams...
Removed 340 seams...
Removed 350 seams...
Removed 360 seams...
Removed 370 seams...
Removed 380 seams...
Removed 390 seams...
Removed 400 seams...
Removed 410 seams...
Removed 420 seams...
Removed 430 s